# Determinants - Exercises

12 exercises covering [notes.md](notes.md) and [theory.ipynb](theory.ipynb).

**Format**:
- Problem: mathematical target plus AI framing when it helps
- Your Solution: scaffold cell for your work
- Solution: full worked answer with checks

| Level | Meaning | Exercises |
|---|---|---|
| * | Core determinant mechanics | 1-4 |
| ** | Structure and spectral links | 5-8 |
| *** | Probability, flows, and low-rank ML uses | 9-12 |

| Topic | Exercises |
|---|---|
| Geometry and direct computation | 1, 2 |
| Characteristic polynomial and adjugate | 3, 4 |
| SPD, conditioning, and special classes | 5, 6 |
| Determinant identities | 7, 8 |
| Log-det in ML | 9, 10 |
| Diversity and low-rank structure | 11, 12 |

In [3]:
import numpy as np

np.set_printoptions(precision=6, suppress=True)


def header(title):
    print("\n" + "=" * len(title))
    print(title)
    print("=" * len(title))


def check_close(name, got, expected, tol=1e-8):
    ok = np.allclose(got, expected, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} - {name}")
    if not ok:
        print('expected:', expected)
        print('got     :', got)
    return ok


def check_true(name, cond):
    print(f"{'PASS' if cond else 'FAIL'} - {name}")
    return cond

---

## Exercise 1: Signed Area and Orientation *

For two vectors in the plane, the determinant of the matrix with those vectors as columns gives the signed area of the spanned parallelogram.

**Task**:
- Implement `signed_area_2d(u, v)`
- Return both the signed area and the orientation label: `positive`, `negative`, or `degenerate`
- Test it on one positively oriented pair, one negatively oriented pair, and one dependent pair

In [4]:
# Exercise 1: Your Solution

def signed_area_2d(u, v):
    u = np.array(u, dtype=float)
    v = np.array(v, dtype=float)
    # YOUR CODE HERE
    pass


pairs = [
    (np.array([3.0, 1.0]), np.array([1.0, 2.0])),
    (np.array([3.0, 1.0]), np.array([1.0, -2.0])),
    (np.array([2.0, 1.0]), np.array([4.0, 2.0])),
]

for u, v in pairs:
    print(signed_area_2d(u, v))

None
None
None


In [11]:
# Exercise 1: Solution

def signed_area_2d(u, v):
    u = np.array(u, dtype=float)
    v = np.array(v, dtype=float)
    area = u[0] * v[1] - u[1] * v[0]
    if np.isclose(area, 0.0):
        label = 'degenerate'
    elif area > 0:
        label = 'positive'
    else:
        label = 'negative'
    return area, label


header('Exercise 1 checks')
a1 = signed_area_2d([3, 1], [1, 2])
a2 = signed_area_2d([3, 1], [1, -2])
a3 = signed_area_2d([2, 1], [4, 2])
print(a1)
print(a2)
print(a3)
check_true('positive orientation detected', a1[1] == 'positive')
check_true('negative orientation detected', a2[1] == 'negative')
check_true('dependent pair detected', a3[1] == 'degenerate')
print('\nTakeaway: determinant sign tracks orientation, and absolute value tracks area.')


Exercise 1 checks
(np.float64(5.0), 'positive')
(np.float64(-7.0), 'negative')
(np.float64(0.0), 'degenerate')
PASS - positive orientation detected
PASS - negative orientation detected
PASS - dependent pair detected

Takeaway: determinant sign tracks orientation, and absolute value tracks area.


---

## Exercise 2: Computation Strategy *

Choose the best determinant method for each matrix: direct formula, triangular-product rule, or elimination.

**Task**:
- Compute each determinant
- State which method is most natural and why
- Identify which matrix is singular

In [12]:
# Exercise 2: Your Solution

A = np.array([[5.0, 3.0], [2.0, 4.0]])
B = np.array([[2.0, 1.0, 0.0], [0.0, 3.0, 1.0], [0.0, 0.0, 4.0]])
C = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]])
D = np.array([[1.0, 0.0, 0.0, 0.0], [2.0, 3.0, 0.0, 0.0], [4.0, 5.0, 6.0, 0.0], [7.0, 8.0, 9.0, 10.0]])

# YOUR CODE HERE
print(np.linalg.det(A), np.linalg.det(B), np.linalg.det(C), np.linalg.det(D))

13.999999999999996 23.999999999999993 -9.51619735392994e-16 180.0


In [7]:
# Exercise 2: Solution

header('Exercise 2 checks')
det_A = np.linalg.det(A)    # 2x2 closed form is natural
det_B = np.linalg.det(B)    # upper triangular -> product of diagonal
det_C = np.linalg.det(C)    # singular dense 3x3
det_D = np.linalg.det(D)    # lower triangular -> product of diagonal

print(f'det(A) = {det_A:.6f}  (2x2 formula)')
print(f'det(B) = {det_B:.6f}  (triangular product)')
print(f'det(C) = {det_C:.6f}  (dense; elimination or Sarrus)')
print(f'det(D) = {det_D:.6f}  (triangular product)')
check_true('C is singular', np.isclose(det_C, 0.0))
check_true('B is nonsingular', not np.isclose(det_B, 0.0))
check_true('D is nonsingular', not np.isclose(det_D, 0.0))
print('\nTakeaway: structure should decide the computation path before you start calculating.')


Exercise 2 checks
det(A) = 14.000000  (2x2 formula)
det(B) = 24.000000  (triangular product)
det(C) = -0.000000  (dense; elimination or Sarrus)
det(D) = 180.000000  (triangular product)
PASS - C is singular
PASS - B is nonsingular
PASS - D is nonsingular

Takeaway: structure should decide the computation path before you start calculating.


---

## Exercise 3: Characteristic Polynomial and Cayley-Hamilton *

For
$$
A = \begin{pmatrix}4 & 2 \\ 1 & 3\end{pmatrix},
$$
compute the characteristic polynomial, the eigenvalues, and verify Cayley-Hamilton.

In [8]:
# Exercise 3: Your Solution

A = np.array([[4.0, 2.0], [1.0, 3.0]])

# YOUR CODE HERE
# Hint: for a 2x2 matrix, p(lambda) = lambda^2 - tr(A) lambda + det(A)
trace_A = None
det_A = None
eigvals = None
cayley_residual = None

print(trace_A, det_A, eigvals, cayley_residual)

None None None None


In [9]:
# Exercise 3: Solution

trace_A = np.trace(A)
det_A = np.linalg.det(A)
eigvals, eigvecs = np.linalg.eig(A)
cayley_residual = A @ A - trace_A * A + det_A * np.eye(2)

header('Exercise 3 checks')
print('trace(A) =', trace_A)
print('det(A)   =', det_A)
print('eigenvalues =', eigvals)
print('Cayley-Hamilton residual =\n', cayley_residual)
check_true('sum of eigenvalues matches trace', np.isclose(np.sum(eigvals), trace_A))
check_true('product of eigenvalues matches determinant', np.isclose(np.prod(eigvals), det_A))
check_close('Cayley-Hamilton', cayley_residual, np.zeros((2, 2)))
print('\nTakeaway: determinant is the constant-term spectral summary in the characteristic polynomial.')


Exercise 3 checks
trace(A) = 7.0
det(A)   = 10.000000000000002
eigenvalues = [5. 2.]
Cayley-Hamilton residual =
 [[0. 0.]
 [0. 0.]]
PASS - sum of eigenvalues matches trace
PASS - product of eigenvalues matches determinant
PASS - Cayley-Hamilton

Takeaway: determinant is the constant-term spectral summary in the characteristic polynomial.


---

## Exercise 4: Cofactors, Adjugate, and the Inverse *

For
$$
A = \begin{pmatrix}1 & 2 & 0 \\ 3 & 1 & 1 \\ 0 & 2 & 1\end{pmatrix},
$$
compute the cofactor matrix, the adjugate, and reconstruct the inverse using
$$
A^{-1} = \frac{\operatorname{adj}(A)}{\det(A)}.
$$

In [10]:
# Exercise 4: Your Solution

A = np.array([[1.0, 2.0, 0.0], [3.0, 1.0, 1.0], [0.0, 2.0, 1.0]])

def cofactor_matrix(A):
    # YOUR CODE HERE
    pass

C = cofactor_matrix(A)
adj = C.T
print(C)
print(adj)

AttributeError: 'NoneType' object has no attribute 'T'

In [ ]:
# Exercise 4: Solution

def cofactor_matrix(A):
    A = np.array(A, dtype=float)
    n = A.shape[0]
    C = np.zeros_like(A)
    for i in range(n):
        for j in range(n):
            sub = np.delete(np.delete(A, i, axis=0), j, axis=1)
            C[i, j] = ((-1) ** (i + j)) * np.linalg.det(sub)
    return C

C = cofactor_matrix(A)
adj = C.T
det_A = np.linalg.det(A)
A_inv = adj / det_A

header('Exercise 4 checks')
print('cofactor matrix =\n', C)
print('\nadjugate =\n', adj)
print('\ninverse from adjugate =\n', A_inv)
check_close('A @ adj(A) = det(A) I', A @ adj, det_A * np.eye(3))
check_close('adjugate inverse matches NumPy inverse', A_inv, np.linalg.inv(A))
print('\nTakeaway: cofactors are not just hand-computation artifacts; they encode the inverse identity.')

---

## Exercise 5: SPD Matrices and Stable Log-Det **

For covariance matrices, the numerically correct quantity is almost always the **log-determinant**, preferably computed through Cholesky rather than direct determinant evaluation.

In [ ]:
# Exercise 5: Your Solution

Sigma = np.array([[4.0, 2.0, 0.0], [2.0, 5.0, 1.0], [0.0, 1.0, 3.0]])

# YOUR CODE HERE
L = None
logdet = None

print(L)
print(logdet)

NameError: name 'np' is not defined

In [ ]:
# Exercise 5: Solution

L = np.linalg.cholesky(Sigma)
logdet = 2.0 * np.sum(np.log(np.diag(L)))
eigvals = np.linalg.eigvalsh(Sigma)
logdet_eigs = np.sum(np.log(eigvals))

header('Exercise 5 checks')
print('Cholesky factor L =\n', L)
print('\nlog det via Cholesky =', logdet)
print('log det via eigenvalues =', logdet_eigs)
check_close('two log-det computations agree', logdet, logdet_eigs)
check_true('SPD determinant is positive', np.linalg.det(Sigma) > 0)
print('\nTakeaway: SPD structure turns determinant work into stable diagonal work after factorization.')

---

## Exercise 6: Determinant Magnitude vs Conditioning **

A determinant can be tiny even when a matrix is perfectly well-conditioned. This exercise is about separating exact singularity logic from numerical stability logic.

In [ ]:
# Exercise 6: Your Solution

A = 0.1 * np.eye(50)
B = np.array([[1.0, 1.0], [1.0, 1.000001]])

# YOUR CODE HERE
print(np.linalg.det(A), np.linalg.cond(A))
print(np.linalg.det(B), np.linalg.cond(B))

NameError: name 'np' is not defined

In [ ]:
# Exercise 6: Solution

det_A = np.linalg.det(A)
cond_A = np.linalg.cond(A)
det_B = np.linalg.det(B)
cond_B = np.linalg.cond(B)
sign_A, logabs_A = np.linalg.slogdet(A)

header('Exercise 6 checks')
print(f'det(0.1 I_50) = {det_A:.6e}')
print(f'cond(0.1 I_50) = {cond_A:.6f}')
print(f'slogdet(0.1 I_50) = (sign={sign_A}, logabsdet={logabs_A:.6f})')
print(f'\ndet(B) = {det_B:.6e}')
print(f'cond(B) = {cond_B:.6e}')
check_true('scaled identity is perfectly conditioned', np.isclose(cond_A, 1.0))
check_true('B is far more ill-conditioned than 0.1I', cond_B > cond_A * 1e3)
print('\nTakeaway: determinant magnitude alone is not a numerical conditioning diagnostic.')

---

## Exercise 7: Matrix Determinant Lemma **

Use the rank-1 determinant update formula
$$
\det(A + uv^T) = (1 + v^T A^{-1} u)\det(A)
$$
to avoid recomputing a full determinant from scratch.

In [ ]:
# Exercise 7: Your Solution

A = np.diag([2.0, 3.0, 4.0])
u = np.array([[1.0], [0.0], [1.0]])
v = np.array([[1.0], [0.0], [1.0]])

# YOUR CODE HERE
lhs = None
rhs = None
print(lhs, rhs)

In [ ]:
# Exercise 7: Solution

lhs = np.linalg.det(A + u @ v.T)
rhs = (1.0 + (v.T @ np.linalg.inv(A) @ u).item()) * np.linalg.det(A)

header('Exercise 7 checks')
print('lhs =', lhs)
print('rhs =', rhs)
check_true('determinant lemma matches direct computation', np.isclose(lhs, rhs))
print('\nTakeaway: low-rank updates let you replace a big determinant with a tiny correction factor.')

---

## Exercise 8: Sylvester and Schur Complement **

Verify two determinant identities that matter for block structure and rectangular products.

In [ ]:
# Exercise 8: Your Solution

A_rect = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
B_rect = A_rect.T

A_block = np.array([[4.0, 2.0], [2.0, 3.0]])
B_block = np.array([[1.0], [0.0]])
D_block = np.array([[2.0]])

# YOUR CODE HERE
print('fill in the computations')

In [ ]:
# Exercise 8: Solution

lhs_syl = np.linalg.det(np.eye(3) + A_rect @ B_rect)
rhs_syl = np.linalg.det(np.eye(2) + B_rect @ A_rect)

M = np.block([[A_block, B_block], [B_block.T, D_block]])
schur = D_block - B_block.T @ np.linalg.inv(A_block) @ B_block
lhs_schur = np.linalg.det(M)
rhs_schur = np.linalg.det(A_block) * np.linalg.det(schur)

header('Exercise 8 checks')
print('Sylvester lhs =', lhs_syl)
print('Sylvester rhs =', rhs_syl)
print('\nSchur lhs =', lhs_schur)
print('Schur rhs =', rhs_schur)
check_true('Sylvester identity holds', np.isclose(lhs_syl, rhs_syl))
check_true('Schur complement identity holds', np.isclose(lhs_schur, rhs_schur))
print('\nTakeaway: block and rectangular structure can move determinant work to smaller matrices.')

---

## Exercise 9: Coupling-Layer Log-Det ***

In flow models, the point is not to make the determinant mathematically interesting. The point is to make it computationally cheap.

Write the Jacobian of a small triangular coupling transform and verify that its log-determinant is just the sum of log-diagonal terms.

In [ ]:
# Exercise 9: Your Solution

def s1(z1):
    return 0.5 * z1


def s2(z1, z2):
    return 0.25 * (z1 + z2)


z = np.array([0.7, -0.2, 1.1])

# YOUR CODE HERE
J = None
logdet = None
print(J)
print(logdet)

In [ ]:
# Exercise 9: Solution

J = np.array([
    [1.0, 0.0, 0.0],
    [0.0, np.exp(s1(z[0])), 0.0],
    [0.0, 0.0, np.exp(s2(z[0], z[1]))],
])
logdet_direct = np.log(abs(np.linalg.det(J)))
logdet_diag = np.log(abs(J[0, 0])) + np.log(abs(J[1, 1])) + np.log(abs(J[2, 2]))
logdet_formula = s1(z[0]) + s2(z[0], z[1])

header('Exercise 9 checks')
print('J =\n', J)
print('logdet direct  =', logdet_direct)
print('logdet diagonal=', logdet_diag)
print('logdet formula =', logdet_formula)
check_true('triangular log-det equals diagonal sum', np.isclose(logdet_direct, logdet_diag))
check_true('closed-form log-det equals direct', np.isclose(logdet_direct, logdet_formula))
print('\nTakeaway: triangular Jacobians are the reason coupling/autoregressive flows are tractable.')

---

## Exercise 10: Gaussian Log-Likelihood and a Tiny GP Term ***

Compute one Gaussian log-likelihood and one tiny Gaussian-process log marginal likelihood term using stable Cholesky-based formulas.

In [ ]:
# Exercise 10: Your Solution

Sigma = np.array([[4.0, 2.0, 0.0], [2.0, 5.0, 1.0], [0.0, 1.0, 3.0]])
x = np.array([1.0, -1.0, 2.0])
mu = np.zeros(3)

K = np.array([[1.0, 0.8, 0.2], [0.8, 1.0, 0.6], [0.2, 0.6, 1.0]]) + 1e-4 * np.eye(3)
y = np.array([1.0, 2.0, 3.0])

# YOUR CODE HERE
gaussian_loglik = None
gp_log_marginal = None
print(gaussian_loglik, gp_log_marginal)

In [13]:
# Exercise 10: Solution

L = np.linalg.cholesky(Sigma)
alpha = np.linalg.solve(L.T, np.linalg.solve(L, x - mu))
gaussian_loglik = -0.5 * (
    len(x) * np.log(2 * np.pi)
    + 2.0 * np.sum(np.log(np.diag(L)))
    + (x - mu) @ alpha
)

Lk = np.linalg.cholesky(K)
alpha_gp = np.linalg.solve(Lk.T, np.linalg.solve(Lk, y))
gp_log_marginal = -0.5 * (
    y @ alpha_gp
    + 2.0 * np.sum(np.log(np.diag(Lk)))
    + len(y) * np.log(2 * np.pi)
)

header('Exercise 10 checks')
print('Gaussian log-likelihood =', gaussian_loglik)
print('Tiny GP log marginal    =', gp_log_marginal)
check_true('Gaussian log-likelihood is finite', np.isfinite(gaussian_loglik))
check_true('GP log marginal is finite', np.isfinite(gp_log_marginal))
print('\nTakeaway: stable probabilistic computation is usually Cholesky + triangular solves + log-diagonal sums.')

NameError: name 'Sigma' is not defined

---

## Exercise 11: DPP-Style Diversity Score ***

Determinants prefer sets of vectors that span large volume. This is why DPPs are used to encourage diversity.

Compare a redundant pair of embeddings with a diverse pair using determinant as the diversity score.

In [ ]:
# Exercise 11: Your Solution

embeddings = np.array([
    [1.0, 0.0],
    [0.95, 0.05],
    [0.0, 1.0],
])
K = embeddings @ embeddings.T

# YOUR CODE HERE
pair_redundant = None
pair_diverse = None
print(pair_redundant, pair_diverse)

None None


In [ ]:
# Exercise 11: Solution

pair_redundant = K[np.ix_([0, 1], [0, 1])]
pair_diverse = K[np.ix_([0, 2], [0, 2])]
score_redundant = np.linalg.det(pair_redundant)
score_diverse = np.linalg.det(pair_diverse)

header('Exercise 11 checks')
print('kernel matrix =\n', K)
print('\nredundant pair determinant =', score_redundant)
print('diverse pair determinant   =', score_diverse)
check_true('diverse pair gets larger determinant score', score_diverse > score_redundant)
print('\nTakeaway: determinant rewards span volume, so it penalizes redundancy automatically.')

---

## Exercise 12: Low-Rank Update and LoRA-Style Structure ***

A LoRA-style update changes a weight matrix by a low-rank term. Use the low-rank determinant identity to verify that a full determinant can be reduced to a small auxiliary determinant.

In [ ]:
# Exercise 12: Your Solution

W = np.diag([2.0, 1.5, 1.2, 0.8])
B = np.array([[1.0, 0.0], [0.5, 1.0], [0.0, 1.0], [1.0, -0.5]])
A = np.array([[0.2, -0.1], [0.1, 0.3], [0.0, 0.2], [-0.2, 0.1]])

# YOUR CODE HERE
lhs = None
rhs = None
print(lhs, rhs)

None None


In [14]:
# Exercise 12: Solution

update = B @ A.T
lhs = np.linalg.det(W + update)
rhs = np.linalg.det(W) * np.linalg.det(np.eye(2) + A.T @ np.linalg.inv(W) @ B)
rank_update = np.linalg.matrix_rank(update)

header('Exercise 12 checks')
print('rank(update) =', rank_update)
print('full determinant     =', lhs)
print('low-rank identity rhs=', rhs)
check_true('update rank <= 2', rank_update <= 2)
check_true('low-rank determinant identity holds', np.isclose(lhs, rhs))
print('\nTakeaway: low-rank structure changes the cost model of determinant computations completely.')

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 2 is different from 3)

---

## What to Review After Finishing

- Can you explain determinant as geometry before writing any formula?
- Do you know when to use cofactor expansion versus elimination?
- Can you separate exact invertibility from numerical conditioning?
- Can you derive eigenvalue relations from `det(lambda I - A)`?
- Do you understand why flow models are built around cheap Jacobian log-determinants?
- Can you recognize when low-rank or block structure makes determinant identities useful?

If you can do those six things cleanly, the chapter is doing its job.

References used in the chapter and notebook design:
- [notes.md](notes.md)
- [theory.ipynb](theory.ipynb)
- [MIT 18.06 Linear Algebra](https://web.mit.edu/18.06/www/)
- [Stanford EE263](https://stanford.edu/class/ee263/)
- [Real NVP](https://arxiv.org/abs/1605.08803)
- [Glow](https://arxiv.org/abs/1807.03039)
- [FFJORD](https://arxiv.org/abs/1810.01367)
- [GPyTorch](https://proceedings.neurips.cc/paper/2018/hash/27e8e17134dd7083b050476733207ea1-Abstract.html)
- [Determinantal Point Processes for Machine Learning](https://www.nowpublishers.com/article/Details/MAL-044)